In [1]:
import numpy as np

# **ufuncs** (funciones universales) 
  
Si el **Broadcasting** define cómo se alinean los datos, las **ufuncs** definen cómo se operan esos datos con una velocidad asombrosa.

---

## 🚀 ¿Qué es exactamente una ufunc?

Una **ufunc** es una función que realiza operaciones **elemento a elemento** sobre los arrays de NumPy. Lo que las hace especiales es que están escritas en **C**, lo que permite que el procesamiento ocurra fuera de la "lentitud" tradicional de los bucles `for` de Python.

### Características clave:

* **Vectorización:** Aplican una operación sobre todo un bloque de datos sin que tú tengas que escribir un bucle.
* **Broadcasting:** Utilizan las reglas de compatibilidad que acabamos de ver para operar arrays de distintas formas.
* **Tipado Dinámico de Salida:** Pueden recibir una lista y devolver un array de NumPy automáticamente.

---

## 🛠️ Tipos de ufuncs

Podemos dividirlas principalmente en dos categorías según cuántas entradas procesan:

| Tipo | Descripción | Ejemplo Matemático | Función en NumPy |
| --- | --- | --- | --- |
| **Unarias** | Toman una sola entrada. | $f(x)$ | `np.sqrt()`, `np.exp()`, `np.sin()` |
| **Binarias** | Toman dos entradas. | $f(x, y)$ | `np.add()`, `np.multiply()`, `np.power()` |


---

## 💡 ¿Por qué son vitales para tu flujo de trabajo?

Como trabajas con **Data Pipelines** y automatización, la eficiencia es tu moneda de cambio. Procesar millones de registros de forma ineficiente puede saturar rápidamente cualquier hardware.

### 1. El fin del bucle `for`

Imagina que quieres calcular el logaritmo de 1 millón de registros de sensores en un proceso de **ETL**.

* **Python puro:** Tienes que iterar 1,000,000 de veces. Cada iteración Python debe verificar el tipo de dato, buscar la función y ejecutarla.
* **ufunc:** NumPy le dice al procesador: *"Aquí tienes un bloque continuo de memoria (float64), aplícale esta operación matemática de C a todos ahora mismo"*.

### 2. Flexibilidad total

Las ufuncs no solo sirven para matemáticas simples; NumPy tiene más de 60 funciones universales que cubren:

* **Aritmética:** `add`, `subtract`, `divide`, `floor_divide`, `mod`.
* **Trigonométricas:** `sin`, `cos`, `arctan`.
* **Comparación:** `greater`, `less`, `equal` (estas devuelven arrays de booleanos, ideales para filtrado).
* **Lógicas:** `logical_and`, `logical_or`.

---

## 🎓 Ejemplo práctico: Normalización de Datos

En un rol de analista, a menudo necesitarás escalar datos para que estén entre 0 y 1 (Min-Max Scaling). Esto se hace usando exclusivamente ufuncs y broadcasting:
  
$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

```python
import numpy as np

# Datos de ventas (ejemplo)
ventas = np.array([100, 250, 400, 600, 900])

# NumPy detecta que min() y max() son agregaciones, 
# y luego aplica ufuncs binarias (resta y división) con broadcasting.
ventas_norm = (ventas - ventas.min()) / (ventas.max() - ventas.min())

print(ventas_norm)
# Output: [0.   0.1875 0.375  0.625  1. ]

```

#### ¿Para qué sirve la **Normalización Min-Max**?
Para un Analista de datos, enfocado en **transformar datos en insights**, esto es vital por tres razones:
1. Comparar **"Manzanas con Naranjas"**: Imagina que quieres comparar Ventas (pesos) con Satisfacción del Cliente (escala 1-5). Como tienen escalas distintas, no puedes compararlas directamente. Si normalizas ambas a $[0, 1]$, puedes ver fácilmente si los picos de ventas coinciden con los picos de satisfacción.

2. Preparación para **Machine Learning**: Muchos algoritmos de IA (como los que podrías correr en Databricks) funcionan mucho mejor si los datos están en una escala pequeña. Si le das números de 0 a 1, el modelo aprende más rápido que si le das números de 0 a 1,000,000.

3. **Visualización de datos**: Es más sencillo crear tableros en Power BI o gráficas comparativas cuando todos tus indicadores comparten la misma unidad de medida relativa.

---

## ⚙️ Parámetros Clave en las ufuncs

A continuación, se resumen los parámetros más utilizados para ajustar el comportamiento de las funciones universales:

| Parámetro | Descripción | Caso de Uso |
| --- | --- | --- |
| **`dtype`** | Define el tipo de dato del resultado final (ej. pasar de  a ). | Ahorro de memoria. |
| **`out`** | Especifica un array existente donde se guardará el resultado. | Evitar la creación de nuevos objetos y saturar la RAM. |
| **`where`** | Una máscara booleana que indica sobre qué elementos aplicar la operación. | Procesamiento condicional sin necesidad de filtros previos. |
| **`casting`** | Controla las reglas de conversión entre tipos de datos. | Seguridad al transformar datos sensibles o de gran precisión. |

---

## 💎 Profundizando en `dtype` y `casting`

Como Analista de Datos, entenderás que no siempre necesitamos la máxima precisión para todos los cálculos. Reducir el tamaño de los datos es una técnica de optimización de nivel profesional.

### El parámetro `dtype`

Por defecto, NumPy suele usar $float64$ o $int64$. Si tus datos de ventas no superan los 2 mil millones, puedes usar $int32$  para reducir el uso de memoria a la mitad.

```python
import numpy as np

arr = np.array([10, 20, 30], dtype='int64')

# Sumamos 5 pero forzamos el resultado a int32
resultado = np.add(arr, 5, dtype='int32')

print(resultado.dtype) # int32

```

### El parámetro `casting`

Cuando cambias tipos de datos, NumPy te protege de perder información mediante el parámetro `casting`. Los valores posibles son:

* **'no'**: No se permite ninguna conversión.
* **'safe'**: Solo permite conversiones que no pierdan valores (ej. de  a ).
* **'same_kind'**: Permite conversiones dentro del mismo tipo (ej. entre diferentes precisiones de ).
* **'unsafe'**: Permite cualquier conversión, incluso si se truncan datos.

---

## 🛠️ Uso de `where` para Pipelines Condicionales

En lugar de filtrar un dataset y luego operar (lo que crea copias intermedias), puedes usar `where` para aplicar la lógica en un solo paso.

```python
import numpy as np

datos = np.array([10, -5, 20, -1, 30])

# Calculamos la raíz cuadrada SOLO de los números positivos
# Los negativos mantendrán su valor original o el que definamos en 'out'
resultado = np.sqrt(datos, where=(datos > 0))

```

---

## 💡 Estrategia de Ingeniería: El Buffer `out`

Para alguien enfocado en transformar datos en insights claros, la gestión de recursos es clave. Usar el parámetro `out` permite reutilizar el espacio físico en la memoria RAM.

1. **Sin `out**`: `c = a + b` (Crea un nuevo array `c` en la RAM).
2. **Con `out**`: `np.add(a, b, out=a)` (Sobrescribe `a`, usando **0 bytes** de memoria adicional).

---

¿Te gustaría que realicemos un **"Desafío de Memoria"**? Podríamos crear un array gigante y medir cuántos megabytes de RAM ahorramos al manipularlo usando exclusivamente estos parámetros y tipos de datos optimizados.